In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
!pip install plotly
!pip install cufflinks

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [24]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# green pallette
GREEN_SCALE = ["#DCE8DF", "#8FAF9D", "#315C48"]   # light -> dark

# One theme for every chart
px.defaults.template = "plotly_white"

In [4]:
df_clean = pd.read_csv('../Dataset/cleaned_retail.csv')

df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M')

Q1 — Geographical Sales Performance

Which countries contribute the most to the retailer's overall revenue, and how does purchasing activity vary across different countries?

In [5]:
country_revenue = (
    df_clean.groupby('Country')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

In [25]:
df_country = country_revenue.reset_index()
df_country.columns = ["Country", "Revenue"]
df_country = df_country.sort_values("Revenue")      

fig = px.bar(
    df_country,
    x="Revenue", y="Country",
    orientation="h",
    text_auto=".3s",                                # shows 1.2M instead of 1200000
    color="Revenue",
    color_continuous_scale=GREEN_SCALE,
    title="Top 10 Countries by Revenue",
)
fig.update_traces(hovertemplate="<b>%{y}</b><br>Revenue: £%{x:,.0f}<extra></extra>")
fig.update_layout(coloraxis_showscale=False, xaxis_title="Revenue (£)", yaxis_title="")
fig.show()

### Key Observations
- The United Kingdom contributes the highest revenue, significantly exceeding all other countries.
- EIRE and the Netherlands are the next-largest contributors.
- Revenue contribution from other European markets is considerably lower.

### Business Insights
The retailer's revenue is heavily concentrated in the UK market, indicating strong dependence on its domestic customer base. International markets represent potential opportunities for geographical diversification.

### Business Recommendations
- Maintain customer retention and marketing efforts in the UK.
- Investigate growth opportunities in the Netherlands, Germany, France and EIRE.
- Compare international markets using order frequency, average order value and customer retention.
- Evaluate shipping and operational costs before expanding into new markets.

Q2. Product Performance

Which products generate the highest revenue and sales volume, and how does product demand vary across the product portfolio?

In [7]:
product_revenue = (
    df_clean.groupby('Description')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

In [26]:
df_product = product_revenue.reset_index()
df_product.columns = ["Product", "Revenue"]
df_product = df_product.sort_values("Revenue")

fig = px.bar(
    df_product,
    x="Revenue", y="Product",
    orientation="h",
    text_auto=".3s",
    color="Revenue",
    color_continuous_scale=GREEN_SCALE,
    title="Top 10 Products by Revenue",
)
fig.update_traces(hovertemplate="<b>%{y}</b><br>Revenue: £%{x:,.0f}<extra></extra>")
fig.update_layout(coloraxis_showscale=False, xaxis_title="Revenue (£)", yaxis_title="", height=500)
fig.show()

> **Data Limitation:** The revenue figures include returns, cancellations, and exceptional transactions. Therefore, the results represent preliminary transaction-level revenue rather than final paid-sales revenue. Interpret product rankings with this limitation in mind.

### Key Observations
- Regency Cakestand 3 Tier is the highest-revenue physical product among the top entries.
- DOTCOM Postage also contributes substantial revenue but represents a service rather than a physical product.
- White Hanging Heart T-Light Holder is another major revenue contributor.
- Product revenue and quantity sold do not necessarily follow the same ranking.

### Business Insights
Products generating higher revenue are not necessarily those selling the highest number of units. Selling price and purchasing volume both influence revenue contribution.

### Business Recommendations
- Prioritise inventory availability for high-performing physical products.
- Evaluate product profitability rather than relying exclusively on revenue.
- Analyse returns and customer purchasing patterns for major products.
- Separate postage and other service-related entries from physical product comparisons.

Q3. Customer Contribution & Behaviour

What proportion of revenue is contributed by the top customers, and how do customer purchasing patterns and order frequency vary?

In [9]:
customer_revenue = (
    df_clean.dropna(subset=['Customer ID'])
    .groupby('Customer ID')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

In [10]:
total_customer_revenue = (
    df_clean.dropna(subset=['Customer ID'])
    ['Revenue'].sum()
)

customer_percentage = (
    customer_revenue / total_customer_revenue
) * 100

cumulative_percentage = customer_percentage.cumsum()

In [ ]:
customers = customer_revenue.index.astype(str)       # treat IDs as text

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Bars (left axis)
fig.add_trace(
    go.Bar(x=customers, y=customer_revenue.values,
           name="Revenue (£)", marker_color="#8FAF9D"),
    secondary_y=False,
)

# Line (right axis)
fig.add_trace(
    go.Scatter(x=customers, y=cumulative_percentage.values,
               name="Cumulative %", mode="lines+markers",
               line=dict(color="#315C48", width=2)),
    secondary_y=True,
)

# 80% reference line 
fig.add_hline(y=80, line_dash="dash", line_color="gray",
              annotation_text="80%", secondary_y=True)

fig.update_layout(title="Customer Revenue Contribution - Pareto Analysis",
                  hovermode="x unified", legend=dict(orientation="h", y=1.1))
fig.update_xaxes(type="category", title_text="Customer ID")
fig.update_yaxes(title_text="Revenue (£)", secondary_y=False)
fig.update_yaxes(title_text="Cumulative Revenue (%)", range=[0, 105], secondary_y=True)
fig.show()

### Key Observations
- The highest-revenue customer contributes approximately 3.5% of identified-customer revenue.
- The top two customers contribute approximately 6.7%.
- The top 10 customers collectively contribute approximately 16.3%.

### Business Insights
Although a small group of customers contributes substantial revenue, the retailer's identified-customer revenue is not overwhelmingly dependent on its top 10 customers. This indicates a broader customer revenue base.

### Business Recommendations
- Develop retention strategies for high-value customers.
- Identify customers with declining purchase frequency.
- Investigate purchasing patterns of frequent and occasional customers.
- Introduce targeted promotions to encourage repeat purchases.

Q4. Seasonal Sales Trends

How does purchasing activity vary over time, and are there identifiable seasonal patterns in monthly revenue?

In [13]:
monthly_revenue = (
    df_clean.groupby(
        df_clean['InvoiceDate'].dt.to_period('M')
    )['Revenue']
    .sum()
)

In [36]:
print(df_clean["InvoiceDate"].max())

2011-12-09 12:50:00


In [32]:
df_month = monthly_revenue.reset_index()
df_month.columns = ["Month", "Revenue"]
df_month["Month"] = pd.to_datetime(df_month["Month"].astype(str)) 
df_month = df_month.iloc[:-1]  # fixes Period type issues

fig = px.line(
    df_month,
    x="Month", y="Revenue",
    markers=True,
    title="Monthly Revenue Trend",
)
fig.update_traces(line_color="#315C48",
                  hovertemplate="%{x|%b %Y}<br>Revenue: £%{y:,.0f}<extra></extra>")
fig.update_xaxes(rangeslider_visible=True, dtick="M1", tickformat="%b %Y")
fig.update_layout(yaxis_title="Revenue (£)", xaxis_title="")
fig.show()

### Key Observations
- November records the highest monthly revenue in both 2010 and 2011.
- Revenue increases considerably between September and November.
- January and February generally show comparatively lower revenue.
- December 2011 records lower revenue, but the dataset contains information only until December 9.

### Business Insights
The repeated November revenue peak suggests a potential seasonal purchasing pattern. The retailer may experience increased purchasing activity towards the end of the year.

### Business Recommendations
- Prepare inventory ahead of the September–November period.
- Schedule promotional campaigns before anticipated demand peaks.
- Monitor stock availability for high-demand products.
- Investigate the comparatively lower revenue during January and February.

**Note**:Dec 2011 excluded because the data covers only part of the month.

Q5. Changing Product Demand

Which products demonstrate fluctuating or declining demand over time, and what purchasing patterns or seasonal variations could explain these changes?

In [15]:
selected_products = [
    'ASSORTED COLOUR BIRD ORNAMENT',
    'MEDIUM CERAMIC TOP STORAGE JAR',
    'PAPER CRAFT , LITTLE BIRDIE',
    'WHITE HANGING HEART T-LIGHT HOLDER',
    'WORLD WAR 2 GLIDERS ASSTD DESIGNS'
]

In [16]:
product_data = df_clean[
    (df_clean['Quantity'] > 0) &
    (df_clean['Price'] > 0) &
    (df_clean['Description'].isin(selected_products))
]

monthly_product_sales = product_data.pivot_table(
    index='YearMonth',
    columns='Description',
    values='Quantity',
    aggfunc='sum'
)

In [30]:
data = monthly_product_sales.copy()
data.index = data.index.astype(str)                  # months as text labels

fig = px.imshow(
    data.T,                                          # products on y-axis, months on x-axis
    color_continuous_scale=GREEN_SCALE,
    aspect="auto",
    labels=dict(x="Month", y="Product", color="Demand"),
    title="Monthly Product Demand Analysis",
)
fig.update_xaxes(tickangle=-60)
fig.update_layout(height=550)
fig.show()

### Key Observations
- Assorted Colour Bird Ornament and World War 2 Gliders show fluctuating monthly demand.
- White Hanging Heart T-Light Holder demonstrates relatively strong demand in several months, with lower activity during parts of 2011.
- Medium Ceramic Top Storage Jar shows an unusually large demand spike in January 2011.
- Paper Craft, Little Birdie shows an exceptionally large spike in December 2011.

### Business Insights
Product demand varies across months, and some products exhibit unusual purchasing spikes. These patterns may reflect seasonal demand, bulk purchases or exceptional transactions. Further investigation is required before identifying sustained demand decline.

### Business Recommendations
- Investigate unusually large transactions to distinguish recurring demand from exceptional purchases.
- Compare the same products across corresponding months of different years.
- Adjust inventory planning according to observed demand patterns.
- Evaluate profitability, holding costs and product relevance before considering discontinuation.